In [ ]:
import os
import glob
import numpy as np
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models.inception import inception_v3

from scipy import linalg


In [2]:
def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """Frechet distance between two Gaussians."""
    mu1, mu2 = np.atleast_1d(mu1), np.atleast_1d(mu2)
    sigma1, sigma2 = np.atleast_2d(sigma1), np.atleast_2d(sigma2)

    diff = mu1 - mu2

    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * np.trace(covmean)
    return fid


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load InceptionV3
inception = inception_v3(pretrained=True, transform_input=False)

# Hook into Mixed_7c layer for spatial features
inception.Mixed_7c.register_forward_hook(
    lambda m, i, o: setattr(inception, 'features', o)
)

inception.eval()
inception.to(device)


Using device: cuda


c:\Users\devgo\OneDrive\Desktop\fid\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\devgo\OneDrive\Desktop\fid\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

In [4]:
transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])


In [5]:
def get_features(img_list, transform, model, device):
    feats = []
    for img_path in tqdm(img_list):
        try:
            img = Image.open(img_path).convert('RGB')
        except:
            print(f"Skipping {img_path}")
            continue
        img = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            _ = model(img)
            feat = model.features.squeeze().cpu().numpy()
            # Spatial mean pooling → vector
            feat = feat.reshape(feat.shape[0], -1).mean(axis=1)
        feats.append(feat)
    return np.array(feats)


In [6]:
# Base dataset folder structure
# project/data/processed_images/<class_name>/
#   ├── ISICxxxx.jpg (real)
#   ├── fake1.jpg (fake)

base_path = r"C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\data\\processed_images"

all_real_imgs = []
all_fake_imgs = []

for class_name in os.listdir(base_path):
    class_path = os.path.join(base_path, class_name)
    if not os.path.isdir(class_path):
        continue

    real_imgs = glob.glob(os.path.join(class_path, "ISIC*.jpg"))
    fake_imgs = [f for f in glob.glob(os.path.join(class_path, "*.jpg"))
                 if not os.path.basename(f).startswith("ISIC")]

    all_real_imgs.extend(real_imgs)
    all_fake_imgs.extend(fake_imgs)

print(f"Total real images: {len(all_real_imgs)}")
print(f"Total fake images: {len(all_fake_imgs)}")


Total real images: 10015
Total fake images: 36920


In [7]:
real_feats = get_features(all_real_imgs, transform, inception, device)
fake_feats = get_features(all_fake_imgs, transform, inception, device)

mu1, sigma1 = np.mean(real_feats, axis=0), np.cov(real_feats, rowvar=False)
mu2, sigma2 = np.mean(fake_feats, axis=0), np.cov(fake_feats, rowvar=False)

overall_sfid = calculate_frechet_distance(mu1, sigma1, mu2, sigma2)
print(f"\n✅ Overall sFID: {overall_sfid:.2f}")


100%|██████████| 36920/36920 [07:36<00:00, 80.94it/s]



✅ Overall sFID: 49.49


In [8]:
class_sfid = {}

for class_name in os.listdir(base_path):
    class_path = os.path.join(base_path, class_name)
    if not os.path.isdir(class_path):
        continue

    real_imgs = glob.glob(os.path.join(class_path, "ISIC*.jpg"))
    fake_imgs = [f for f in glob.glob(os.path.join(class_path, "*.jpg"))
                 if not os.path.basename(f).startswith("ISIC")]

    if len(real_imgs) == 0 or len(fake_imgs) == 0:
        continue

    real_feats = get_features(real_imgs, transform, inception, device)
    fake_feats = get_features(fake_imgs, transform, inception, device)

    mu1, sigma1 = np.mean(real_feats, axis=0), np.cov(real_feats, rowvar=False)
    mu2, sigma2 = np.mean(fake_feats, axis=0), np.cov(fake_feats, rowvar=False)

    fid_score = calculate_frechet_distance(mu1, sigma1, mu2, sigma2)
    class_sfid[class_name] = fid_score

print("\n✅ sFID per class:")
for cls, score in class_sfid.items():
    print(f"{cls}: {score:.2f}")


100%|██████████| 6563/6563 [01:22<00:00, 79.89it/s]



✅ sFID per class:
AKIEC: 66.15
BCC: 94.20
BKL: 47.10
DF: 104.44
MEL: 41.11
VASC: 123.41
